# Modelo simple: regla y regresión logística

Compara, sobre `data/processed/features.parquet` (una fila por merchant):

- **Regla:** ordenar por la peor de tres señales: la peor aprobación entre 7 y 30 días, los rechazos del día de la foto, o el peor día de rechazos en los 30 días hasta la foto (`max_rechazos_1d_30d`). No se entrena.
- **Logística A, B y C:** regresión logística con regularización L2, con cada vez más variables. Si C no mejora a B, se queda B.

| Modelo | Variables |
|---|---|
| A | aprobación a 7 días (con respaldo a 30 días e histórica) |
| B | A + aprobación histórica + indicador "sin actividad a 7 días" + log del peor día de rechazos en 30 días |
| C | B + log del ticket promedio a 30 días + log de los intentos a 7 días |

**Validación.** `StratifiedKFold` de 5 folds repetido con 10 semillas. Cada merchant se predice una vez por repetición, con un modelo que no lo vio; las métricas se calculan sobre esas predicciones y se promedian entre repeticiones. Se repite porque con un solo corte el test tiene ~4 positivos y los modelos no se pueden distinguir. El desvío que se informa es el de partir los datos de otra manera, no la incertidumbre de tener 18 casos.

**Métricas.** PR-AUC junto a la prevalencia, y positivos capturados en el top 1% y 5% del score. `MEA1EE5` no está en el dataset: nunca operó. Los empates en el corte se reparten al azar (valor esperado): 753 merchants tienen aprobación exactamente 0, y desempatar por el orden de las filas favorecería a los positivos, que van primero en el dataset.

La regla y la elección de variables salieron de mirar estos mismos 12 positivos; la logística aprende dentro de cada fold, pero las variables candidatas no.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "processed" / "dataset.parquet"
FEATURES_PATH = ROOT / "data" / "processed" / "features.parquet"
OUT_DIR = ROOT / "docs" / "modelo"

N_FOLDS = 5
N_REPEATS = 10
L2_C = 0.1  # regularización fuerte, sin ajustar: con 12 positivos no hay con qué elegirla

data = pd.read_parquet(FEATURES_PATH)
print(f"{len(data):,} merchants, {int(data['target'].sum())} positivos ({int(data.loc[data['group_d'], 'target'].sum())} en el grupo D)")

## Variables

Las tasas con valor −1 (sin transacciones) ya vienen como nulas. Para el ticket y los intentos, nulo significa que no hubo actividad: se toma como 0.

In [ ]:
rate_7d = data["me_txn_approval_rate_7d"].clip(0, 1)
rate_30d = data["me_txn_approval_rate_30d"].clip(0, 1)
rate_life = data["me_lifetime_txn_approval_rate"].clip(0, 1)
recent = pd.concat([rate_7d, rate_30d], axis=1).min(axis=1, skipna=True)
approval = recent.fillna(rate_life)

raw = pd.read_parquet(DATA_PATH, columns=["merchant_id", "me_attempted_txn_sum_1d", "me_approved_txn_sum_1d"])
raw = data[["merchant_id"]].merge(raw, on="merchant_id", how="left", validate="one_to_one")
attempted = raw["me_attempted_txn_sum_1d"].where(raw["me_attempted_txn_sum_1d"] > -0.5)
approved = raw["me_approved_txn_sum_1d"].where(raw["me_approved_txn_sum_1d"] > -0.5)
rejected = (attempted - approved).where(attempted.notna())
rejected.index = data.index

V = pd.DataFrame({
    "aprobacion_7d": approval,
    "aprobacion_historica": rate_life,
    "sin_actividad_7d": rate_7d.isna().astype(float),
    "log_ticket_30d": np.log1p(data["me_attempted_tpv_avg_30d"].fillna(0).clip(lower=0)),
    "log_intentos_7d": np.log1p(data["me_attempted_txn_sum_7d"].fillna(0).clip(lower=0)),
    "log_max_rechazos_30d": np.log1p(data["max_rechazos_1d_30d"].fillna(0).clip(lower=0)),
})
SETS = {
    "A": ["aprobacion_7d"],
    "B": ["aprobacion_7d", "aprobacion_historica", "sin_actividad_7d", "log_max_rechazos_30d"],
    "C": ["aprobacion_7d", "aprobacion_historica", "sin_actividad_7d", "log_ticket_30d", "log_intentos_7d", "log_max_rechazos_30d"],
}
print((V.isna().mean() * 100).round(1).to_string())

## Modelos y métricas

In [ ]:
def logistic():
    return make_pipeline(
        SimpleImputer(strategy="median"), StandardScaler(),
        LogisticRegression(C=L2_C, class_weight="balanced", max_iter=1000),
    )


def captured_in_top(y, score, k):
    """Positivos esperados entre los k de mayor score. Los empates en el corte se reparten al azar:
    no depende del orden de las filas (en el dataset los positivos van primero)."""
    cut = np.sort(score)[::-1][k - 1]
    above = score > cut
    tied = score == cut
    return y[above].sum() + y[tied].sum() * (k - above.sum()) / tied.sum()


def metrics(y, score):
    out = {"pr_auc": average_precision_score(y, score)}
    for frac in (0.01, 0.05):
        out[f"capturados_top{int(frac * 100)}"] = captured_in_top(y, score, int(np.ceil(frac * len(y))))
    return out


def cross_validate(make_model, X, y):
    rows = []
    for seed in range(N_REPEATS):
        oof = np.zeros(len(y))
        for train, test in StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(X, y):
            model = make_model()
            model.fit(X.iloc[train], y[train])
            oof[test] = model.predict_proba(X.iloc[test])[:, 1]
        rows.append(metrics(y, oof))
    return pd.DataFrame(rows)


def risk_from_low(series):
    return 1 - series.rank(method="average", pct=True)


def risk_from_high(series):
    return series.rank(method="average", pct=True)


def run(mask, label):
    y = data.loc[mask, "target"].to_numpy()
    n, npos = len(y), int(y.sum())
    k1, k5 = int(np.ceil(0.01 * n)), int(np.ceil(0.05 * n))
    rows = []

    # Peor de las tres: aprobación baja, rechazos del día de la foto, o el peor día de los 30.
    rule_score = np.fmax(
        np.fmax(
            risk_from_low(approval.loc[mask]).fillna(-1).to_numpy(),
            risk_from_high(rejected.loc[mask]).fillna(-1).to_numpy(),
        ),
        risk_from_high(data.loc[mask, "max_rechazos_1d_30d"]).fillna(-1).to_numpy(),
    )
    rule = metrics(y, rule_score)
    rows.append({"modelo": "Regla (sin entrenar)", **{k: [v] for k, v in rule.items()}})
    results = {"Regla (sin entrenar)": pd.DataFrame([rule])}
    for name, cols in SETS.items():
        results[f"Logística {name}"] = cross_validate(logistic, V.loc[mask, cols].reset_index(drop=True), y)

    table = []
    for model, r in results.items():
        table.append({
            "grupo": label, "modelo": model,
            "pr_auc": r["pr_auc"].mean(), "pr_auc_desvio": r["pr_auc"].std(ddof=0),
            "top1": r["capturados_top1"].mean(), "top5": r["capturados_top5"].mean(),
            "top5_desvio": r["capturados_top5"].std(ddof=0),
        })
    table = pd.DataFrame(table)
    table["prevalencia"] = npos / n
    table["positivos"] = npos
    table["k1"], table["k5"] = k1, k5
    print(f"\n== {label}: {n:,} merchants, {npos} positivos, prevalencia {npos / n:.4%} ==")
    print(f"azar: top 1% = {k1} merchants, ~{npos * k1 / n:.2f} positivos; top 5% = {k5} merchants, ~{npos * k5 / n:.2f} positivos")
    print(table[["modelo", "pr_auc", "pr_auc_desvio", "top1", "top5", "top5_desvio"]].round(4).to_string(index=False))
    return table


all_rows = pd.Series(True, index=data.index)
table_all = run(all_rows, "todos")


In [ ]:
table_no_d = run(~data["group_d"], "sin grupo D")

## Coeficientes (ajustando con todos los datos)

In [ ]:
y_all = data["target"].to_numpy()
for name in ("B", "C"):
    model = logistic().fit(V[SETS[name]], y_all)
    coef = pd.Series(model[-1].coef_[0], index=SETS[name])
    print(f"Logística {name} (coeficientes sobre variables estandarizadas; positivo = más riesgo):")
    print(coef.round(3).to_string())
    print()

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
results = pd.concat([table_all, table_no_d], ignore_index=True)
results.round(5).to_csv(OUT_DIR / "logistica_metricas.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, (label, t) in zip(axes, results.groupby("grupo", sort=False)):
    t = t.reset_index(drop=True)
    ax.bar(t["modelo"], t["top5"], yerr=t["top5_desvio"], color=["#B4B2A9"] + ["#378ADD"] * 3 + ["#D85A30"], capsize=4)
    ax.axhline(t["positivos"][0] * t["k5"][0] / (t["k5"][0] / 0.05), color="#444441", ls="--", label="azar")
    ax.set_title(f"{label}: positivos en el top 5% (de {int(t['positivos'][0])})")
    ax.set_ylabel("positivos capturados")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "logistica.png", dpi=110)
plt.show()